#Test Script Classification

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import pandas as pd
import numpy as np
import re, ast, json
from collections import Counter
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import accuracy_score, classification_report
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay


# Encoders
label_encoder     = joblib.load('label_encoder.pkl')
ohe_lang          = joblib.load('ohe_lang.pkl')
ohe_cat           = joblib.load('ohe_cat.pkl')
mlb_objects       = joblib.load('mlb_objects.pkl')
common_languages  = joblib.load('common_languages.pkl')

# Scaler
scaler            = joblib.load('scaler.pkl')

# Text features
tfidf             = joblib.load('tfidf.pkl')
svd               = joblib.load('svd.pkl')

# Feature info
selected_features = joblib.load('selected_features.pkl')
company_freq_map  = joblib.load('freq_map.pkl')
high_null_cols    = joblib.load('high_null_cols.pkl')
num_medians       = joblib.load('num_medians.pkl')

# Date params
with open('date_params.json', 'r') as f:
    date_params = json.load(f)
median_date = pd.to_datetime(date_params['median_date'])
min_year    = date_params['min_year']

# Models
models = {
    'Decision Tree': joblib.load('decisionTree_model.pkl'),
    'XGBoost':       joblib.load('xgboost_model.pkl'),
    'AdaBoost':      joblib.load('adaboost_model.pkl'),
    'Logistic Regression': joblib.load('LogisticRegression_model.pkl'),
}

print(f" All artifacts loaded!")
print(f"  Features  : {len(selected_features)}")
print(f"  Classes   : {label_encoder.classes_}")
print(f"  Min year  : {min_year}")
print(f"  Median date: {median_date}")

 All artifacts loaded!
  Features  : 156
  Classes   : ['high' 'low' 'medium' 'very low']
  Min year  : 1900
  Median date: 2010-08-27 00:00:00


In [3]:
df_test = pd.read_csv("test_data_class.csv")
print("shape: ", df_test.shape)
print("\n columns:", df_test.columns.tolist())

shape:  (10000, 39)

 columns: ['id', 'title', 'quality', 'theatrical', 'movie_valence', 'movie_vad_valence', 'movie_vad_arousal', 'movie_vad_dominance', 'movie_intensity_anger', 'movie_intensity_anticipation', 'movie_intensity_disgust', 'movie_intensity_fear', 'movie_intensity_joy', 'movie_intensity_sadness', 'movie_intensity_surprise', 'movie_intensity_trust', 'movie_scl_shift', 'movie_scl_coverage', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularityLevel', 'poster_path', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages']


In [4]:
# bool cols
bool_modes = joblib.load('bool_modes.pkl')
for col, mode_val in bool_modes.items():
    if col in df_test.columns:
        df_test[col] = df_test[col].fillna(mode_val).astype(int)

In [5]:

y_true = None
if 'popularityLevel' in df_test.columns:
    y_true = df_test['popularityLevel'].astype(str).str.strip().str.lower()
    y_true = label_encoder.transform(y_true)
    df_test = df_test.drop(columns=['popularityLevel'])
    print(f"Target extracted: {len(y_true)} samples")
else:
    print("No target column found - prediction mode only")
df_test = df_test.drop(columns=[c for c in high_null_cols if c in df_test.columns])

# Drop other useless columns
cols_to_drop = ['backdrop_path', 'homepage', 'tagline', 'poster_path', 'id', 'imdb_id']
df_test = df_test.drop(columns=[c for c in cols_to_drop if c in df_test.columns])

print(f"After dropping columns: {df_test.shape}")


Target extracted: 10000 samples
After dropping columns: (10000, 18)


In [6]:
#HANDLE MISSING VALUES

# Numerical columns
for col in num_medians:
    if col in df_test.columns:
        df_test[col] = df_test[col].fillna(num_medians[col])

# Multi-label columns
multi_label_cols = ['production_companies', 'production_countries', 'spoken_languages', 'genres']
for col in multi_label_cols:
    if col in df_test.columns:
        df_test[col] = df_test[col].fillna("[]")

# Overview
if 'overview' in df_test.columns:
    df_test['overview'] = df_test['overview'].fillna("")

# Adult
adult_map = {"False": 0, "True": 1, False: 0, True: 1, "Unknown": -1}
if 'adult' in df_test.columns:
    df_test['adult'] = df_test['adult'].fillna("Unknown").map(adult_map).astype(int)

print(f"After missing values: {df_test.shape}")

After missing values: (10000, 18)


In [7]:
print(f"\n1. After preprocessing (df_test shape): {df_test.shape}")
print(f"   First 5 rows index: {df_test.index[:5].tolist()}")


1. After preprocessing (df_test shape): (10000, 18)
   First 5 rows index: [0, 1, 2, 3, 4]


##Feature Engineering

In [8]:
#FEATURE ENGINEERING

# Title features
if 'title' in df_test.columns:
    title_text = df_test['title'].fillna("").astype(str)
    df_test['title_word_count'] = title_text.str.split().str.len()
    df_test['title_char_count'] = title_text.str.len()
    df_test['title_has_number'] = title_text.str.contains(r"\d", regex=True).astype(int)
    df_test = df_test.drop(columns=['title', 'original_title'], errors='ignore')
print(" Title features done")

# Date features
if 'release_date' in df_test.columns:
    df_test['release_date'] = pd.to_datetime(df_test['release_date'], errors='coerce')
    df_test['release_date'] = df_test['release_date'].fillna(median_date)

    current_year = 2026
    df_test['release_year']       = df_test['release_date'].dt.year
    df_test['release_month']      = df_test['release_date'].dt.month
    df_test['release_day']        = df_test['release_date'].dt.day
    df_test['release_dayofweek']  = df_test['release_date'].dt.dayofweek
    df_test['release_quarter']    = df_test['release_date'].dt.quarter
    df_test['movie_age']          = current_year - df_test['release_year']
    df_test['release_year_norm']  = df_test['release_year'] - min_year
    df_test['is_weekend_release'] = df_test['release_dayofweek'].isin([5, 6]).astype(int)
    df_test['is_summer']          = df_test['release_month'].isin([6, 7, 8]).astype(int)
    df_test['is_winter']          = df_test['release_month'].isin([12, 1, 2]).astype(int)
    df_test['is_holiday_season']  = df_test['release_month'].isin([11, 12]).astype(int)
    df_test['recency_score']      = 1 / (df_test['movie_age'] + 1)

    df_test = df_test.drop(columns=['release_date'])
print(" Date features done")

# Budget/Revenue features
for col in ['budget', 'runtime', 'vote_count', 'revenue']:
    if col not in df_test.columns:
        df_test[col] = 0

df_test['budget_runtime_ratio'] = df_test['budget'] / (df_test['runtime'] + 1)
df_test['budget_per_vote']      = df_test['budget'] / (df_test['vote_count'] + 1)
df_test['revenue_per_vote']     = df_test['revenue'] / (df_test['vote_count'] + 1)

# Log Transform
log_cols = ['revenue', 'budget', 'vote_count', 'runtime',
            'budget_runtime_ratio', 'budget_per_vote', 'revenue_per_vote']
for col in log_cols:
    if col in df_test.columns:
        df_test[col] = np.log1p(df_test[col].clip(lower=0))
print("Budget features done")

 Title features done
 Date features done
Budget features done


##Encoding

In [9]:
#  ENCODING

# Original Language
if 'original_language' in df_test.columns:
    df_test['original_language'] = df_test['original_language'].apply(
        lambda x: x if x in common_languages else "other"
    )
    lang_test = ohe_lang.transform(df_test[['original_language']])
    lang_cols = ohe_lang.get_feature_names_out(['original_language'])
    df_test = pd.concat([
        df_test.drop(columns=['original_language']),
        pd.DataFrame(lang_test, columns=lang_cols, index=df_test.index)
    ], axis=1)
print(" Language encoding done")

# Quality & Status
if 'quality' in df_test.columns and 'status' in df_test.columns:
    df_test['quality'] = df_test['quality'].astype(str).str.strip().str.lower()
    df_test['status']  = df_test['status'].astype(str).str.strip().str.lower()

    split = df_test['quality'].str.split('_', expand=True)
    df_test['quality_type']  = split[0]
    df_test['quality_level'] = split[1]

    quality_level_map = {'uncertain': 0, 'likely': 1, 'legitimate': 2, 'confident': 3}
    df_test['quality_level'] = df_test['quality_level'].map(quality_level_map).fillna(-1).astype(int)

    cat_test = ohe_cat.transform(df_test[['status', 'quality_type']])
    cat_feat_names = ohe_cat.get_feature_names_out(['status', 'quality_type'])
    df_test = pd.concat([
        df_test.drop(columns=['quality', 'status', 'quality_type'], errors='ignore'),
        pd.DataFrame(cat_test, columns=cat_feat_names, index=df_test.index)
    ], axis=1)
print("Quality/Status encoding done")

# Multi-label columns (genres, production_countries, spoken_languages)
def parse_multilabel(series):
    def parse(x):
        if isinstance(x, list): return [str(i).strip().lower() for i in x]
        if pd.isna(x) or x in ['', 'unknown', '[]']: return []
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return [str(i).strip().lower() for i in parsed]
        except: pass
        return [i.strip().lower() for i in str(x).split(',')]
    return series.apply(parse)

ml_cols = ['genres', 'production_countries', 'spoken_languages']
for col in ml_cols:
    if col in df_test.columns:
        df_test[col] = parse_multilabel(df_test[col])
        mlb = mlb_objects[col]

        top_labels = set(mlb.classes_)
        df_test[col] = df_test[col].apply(
            lambda lst: [x for x in lst if x in top_labels] or ['other']
        )

        te_enc = mlb.transform(df_test[col])
        te_df = pd.DataFrame(te_enc,
                             columns=[f"{col}_{c}" for c in mlb.classes_],
                             index=df_test.index)
        df_test = pd.concat([df_test.drop(columns=[col]), te_df], axis=1)
print(" Multi-label encoding done")

# Production Companies
def parse_production_companies(series):
    def parse(x):
        if isinstance(x, list): return [str(i).strip().lower() for i in x]
        if pd.isna(x) or x in ['', 'unknown', '[]', '{}']: return []
        x = str(x).strip()
        if x.startswith('[') and x.endswith(']'):
            try:
                parsed = ast.literal_eval(x)
                if isinstance(parsed, list):
                    return [str(i).strip().lower() for i in parsed]
            except: pass
        matches = re.findall(r"'([^']*)'|\"([^\"]*)\"", x)
        extracted = [m[0] or m[1] for m in matches if m[0] or m[1]]
        if extracted: return [e.strip().lower() for e in extracted]
        return [i.strip().lower() for i in x.split(',') if i.strip()]
    return series.apply(parse)

if 'production_companies' in df_test.columns:
    df_test['production_companies'] = parse_production_companies(df_test['production_companies'])

    def encode_company_frequency(series, freq_map):
        rows = []
        for companies in series:
            if not companies:
                rows.append([0, 0, 0, 0, 0])
                continue
            values = [freq_map.get(c, 0) for c in companies]
            rows.append([np.mean(values), np.max(values), np.min(values),
                         np.std(values), len(companies)])
        return pd.DataFrame(rows,
            columns=['production_companies_freq_mean', 'production_companies_freq_max',
                     'production_companies_freq_min', 'production_companies_freq_std',
                     'production_companies_count'],
            index=series.index)

    company_features = encode_company_frequency(df_test['production_companies'], company_freq_map)
    df_test = pd.concat([df_test.drop(columns=['production_companies']), company_features], axis=1)
print("Production companies encoding done")

# Count features
for prefix in ['genres', 'production_countries', 'spoken_languages']:
    cols = [c for c in df_test.columns if c.startswith(f"{prefix}_")]
    if cols:
        df_test[f"num_{prefix}"] = df_test[cols].sum(axis=1)

if 'production_companies_count' in df_test.columns:
    df_test['num_companies'] = df_test['production_companies_count']
else:
    df_test['num_companies'] = 0
print("Count features done")

# Overview TF-IDF + SVD
if 'overview' in df_test.columns:
    X_overview_test = tfidf.transform(df_test['overview'].fillna(''))
    X_test_topics = svd.transform(X_overview_test)
    topic_cols = [f"overview_topic_{i}" for i in range(X_test_topics.shape[1])]
    df_test = pd.concat([
        df_test.drop(columns=['overview']),
        pd.DataFrame(X_test_topics, columns=topic_cols, index=df_test.index)
    ], axis=1)
print(" Overview LSA done")

 Language encoding done
Quality/Status encoding done
 Multi-label encoding done
Production companies encoding done
Count features done
 Overview LSA done


In [10]:
# Replace inf values
df_test = df_test.replace([np.inf, -np.inf], 0)

# Drop any object columns remaining
obj_cols = df_test.select_dtypes(include='object').columns.tolist()
if obj_cols:
    print(f"WARNING — dropping object columns: {obj_cols}")
    df_test = df_test.drop(columns=obj_cols)

print(f"After cleanup: {df_test.shape}")

After cleanup: (10000, 156)


In [11]:
for col in selected_features:
    if col not in df_test.columns:
        df_test[col] = 0

df_test = df_test[selected_features]
print(f"Aligned to selected features: {df_test.shape}")

Aligned to selected features: (10000, 156)


In [12]:
print(type(selected_features))   # should be list
print(len(selected_features))    # should be 156

<class 'list'>
156


In [13]:
X_test_scaled = scaler.transform(df_test)
print(f"Scaled shape: {X_test_scaled.shape}")

print("\n" + "=" * 60)
print("RUNNING PREDICTIONS")
print("=" * 60)

predictions = {}
timings = {}

for name, model in models.items():
    start = time.time()
    preds = model.predict(X_test_scaled)
    elapsed = time.time() - start
    predictions[name] = preds
    timings[name] = elapsed
    print(f" {name}: {elapsed:.4f}s")

Scaled shape: (10000, 156)

RUNNING PREDICTIONS
 Decision Tree: 0.0063s
 XGBoost: 0.0180s
 AdaBoost: 3.5708s
 Logistic Regression: 0.0046s


In [14]:
if y_true is not None:
    print("\n" + "=" * 60)
    print("EVALUATION RESULTS")
    print("=" * 60)

    results_summary = []
    for name, preds in predictions.items():
        acc = accuracy_score(y_true, preds)
        results_summary.append({'Model': name, 'Accuracy': acc, 'Time(s)': timings[name]})
        print(f"\n{name}")
        print(f"  Accuracy : {acc:.4f}")
        print(f"  Time     : {timings[name]:.4f}s")
        print(f"  Classes  : {label_encoder.classes_}")

        print(classification_report(y_true, preds, target_names=label_encoder.classes_, zero_division=0))

        # unique_labels = np.unique(np.concatenate((y_true, preds)))
        # current_target_names = label_encoder.inverse_transform(unique_labels)
        # print(classification_report(y_true, preds, labels=unique_labels, target_names=current_target_names, zero_division=0))

    summary_df = pd.DataFrame(results_summary).sort_values('Accuracy', ascending=False)
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(summary_df.to_string(index=False))


    best_model_name = summary_df.iloc[0]['Model']
else:
    best_model_name = list(models.keys())[0]
    print("\nNo ground truth available - predictions only")


EVALUATION RESULTS

Decision Tree
  Accuracy : 0.8516
  Time     : 0.0063s
  Classes  : ['high' 'low' 'medium' 'very low']
              precision    recall  f1-score   support

        high       0.83      0.74      0.78       150
         low       0.54      0.33      0.41      1395
      medium       0.76      0.66      0.70       754
    very low       0.89      0.97      0.93      7701

    accuracy                           0.85     10000
   macro avg       0.76      0.67      0.71     10000
weighted avg       0.83      0.85      0.84     10000


XGBoost
  Accuracy : 0.8568
  Time     : 0.0180s
  Classes  : ['high' 'low' 'medium' 'very low']
              precision    recall  f1-score   support

        high       0.83      0.77      0.80       150
         low       0.57      0.32      0.41      1395
      medium       0.76      0.70      0.73       754
    very low       0.89      0.97      0.93      7701

    accuracy                           0.86     10000
   macro avg     

In [15]:
#Save prediction
best_preds = predictions[best_model_name]
pred_labels = label_encoder.inverse_transform(best_preds)

output = pd.DataFrame({
    'predicted_class_encoded': best_preds,
    'predicted_class_label': pred_labels
})

if y_true is not None:
    output['true_label'] = label_encoder.inverse_transform(y_true)
    output['correct'] = (output['predicted_class_label'] == output['true_label'])

output.to_csv('test_predictions.csv', index=False)
print(f"\nSaved predictions from '{best_model_name}' → test_predictions.csv")
print("\n" + "=" * 60)
print("FIRST 20 PREDICTIONS")
print("=" * 60)
print(output)


Saved predictions from 'XGBoost' → test_predictions.csv

FIRST 20 PREDICTIONS
      predicted_class_encoded predicted_class_label true_label  correct
0                           3              very low   very low     True
1                           3              very low   very low     True
2                           3              very low   very low     True
3                           3              very low   very low     True
4                           3              very low   very low     True
...                       ...                   ...        ...      ...
9995                        2                medium     medium     True
9996                        3              very low   very low     True
9997                        3              very low   very low     True
9998                        3              very low   very low     True
9999                        0                  high       high     True

[10000 rows x 4 columns]
